# Práctica — Clase 11: arregla el "traductor" de Babel Fish

En la clase viste que el proyecto real de Babel Fish (`babel-fish-project/translator-with-voice-and-watsonx/worker.py`) tiene un bug real: la función `watsonx_process_message`, que se supone que traduce el mensaje del usuario, en realidad usa un prompt genérico que no garantiza ninguna traducción:

```python
def watsonx_process_message(user_message):
    prompt = f"""Respond to the query: ```{user_message}```"""
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()
```

En esta práctica vas a resolver ese mismo problema, pero con la vía 100% gratuita que usaste en clase: en vez de depender de un prompt bien escrito para un LLM generalista, vas a construir una función que reciba un texto en inglés y un idioma de destino, y que **garantice** la traducción usando el pipeline de traducción de Hugging Face.

## Ejercicio — Construye un traductor que sí traduzca de verdad

**Contexto real**: el bug de Babel Fish existe porque nadie verificó que el prompt `"Respond to the query: ..."` produjera una traducción real para distintas entradas. Tu tarea es construir una alternativa que no dependa de la buena voluntad de un prompt.

**Pistas**:
- Ya tienes el pipeline `translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")` de la clase, que traduce inglés → español.
- Para traducir a **otros** idiomas de destino, Hugging Face tiene modelos equivalentes con el mismo patrón de nombre: `Helsinki-NLP/opus-mt-en-fr` (francés), `Helsinki-NLP/opus-mt-en-de` (alemán), `Helsinki-NLP/opus-mt-en-it` (italiano), etc. El patrón general es `Helsinki-NLP/opus-mt-en-{codigo_idioma}`.
- Tu función debe recibir el texto en inglés y el idioma de destino (por ejemplo, `"es"`, `"fr"`, `"de"`), y devolver el texto traducido, cargando el modelo correspondiente a ese idioma.
- Para no recargar el mismo modelo varias veces, puedes guardar los pipelines ya cargados en un diccionario (`cache`).

Completa la función `traducir_a_idioma` de abajo — no reutilices la versión "bug" del prompt genérico.

In [ ]:
from transformers import pipeline

# Cache de pipelines ya cargados, para no descargar el mismo modelo dos veces.
_pipelines_cache = {}

def traducir_a_idioma(texto_ingles, idioma_destino):
    """
    Traduce un texto en ingles al idioma de destino indicado, usando un
    modelo de traduccion especializado de Hugging Face (nunca un prompt
    generico que "podria" o no traducir).

    texto_ingles: el texto de entrada, en ingles.
    idioma_destino: codigo de idioma de destino, por ejemplo "es", "fr", "de".

    Debe devolver el texto ya traducido (str).
    """
    # TODO 1: construye el nombre del modelo de Hugging Face para este idioma
    #         de destino, siguiendo el patron "Helsinki-NLP/opus-mt-en-{idioma_destino}"
    nombre_modelo = None  # <-- completa aqui

    # TODO 2: si el pipeline para este idioma ya esta en _pipelines_cache, reutilizalo;
    #         si no, crealo con pipeline(f"translation_en_to_{idioma_destino}", model=nombre_modelo)
    #         y guardalo en _pipelines_cache antes de usarlo.


    # TODO 3: usa el pipeline para traducir texto_ingles y devuelve
    #         unicamente el texto traducido (string), no la lista/diccionario completo.
    return None  # <-- reemplaza esto por el resultado real


# Prueba tu funcion con al menos dos idiomas de destino distintos:
# print(traducir_a_idioma("Where is the nearest train station?", "es"))
# print(traducir_a_idioma("Where is the nearest train station?", "fr"))

<details>
<summary>💡 Ver solución</summary>

```python
from transformers import pipeline

_pipelines_cache = {}

def traducir_a_idioma(texto_ingles, idioma_destino):
    """
    Traduce un texto en ingles al idioma de destino indicado, usando un
    modelo de traduccion especializado de Hugging Face.
    """
    # Construimos el nombre del modelo siguiendo el patron de Helsinki-NLP
    nombre_modelo = f"Helsinki-NLP/opus-mt-en-{idioma_destino}"

    # Reutilizamos el pipeline si ya lo cargamos antes, si no lo creamos
    if idioma_destino not in _pipelines_cache:
        _pipelines_cache[idioma_destino] = pipeline(
            f"translation_en_to_{idioma_destino}",
            model=nombre_modelo,
        )
    translator = _pipelines_cache[idioma_destino]

    # Ejecutamos la traduccion y devolvemos solo el texto
    resultado = translator(texto_ingles)
    return resultado[0]["translation_text"]


print(traducir_a_idioma("Where is the nearest train station?", "es"))
print(traducir_a_idioma("Where is the nearest train station?", "fr"))
```

A diferencia del prompt genérico de Babel Fish (`"Respond to the query: ..."`), esta función no puede "olvidarse" de traducir: el modelo que carga está entrenado específicamente para la tarea de traducción de inglés a un idioma concreto, así que la traducción está garantizada por el propio diseño del modelo, no por la suerte de un prompt bien (o mal) escrito.
</details>

## Para seguir

- Notebook de clase: [clase11_traduccion_babelfish.ipynb](../notebooks_clase/clase11_traduccion_babelfish.ipynb)
- Capítulo del manual: [15_traduccion_babelfish.md](../manual/15_traduccion_babelfish.md)